# Q1: Supervised Learning — Heart Disease Classification

This notebook builds and evaluates classification models to predict whether a patient has heart disease using the `q1_heart_disease.csv` dataset.

## 1. Data Loading and Inspection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/q1_heart_disease.csv')
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

Shape: (800, 12)

Data Types:
age                  int64
sex                  int64
chest_pain_type        str
resting_bp         float64
cholesterol        float64
fasting_bs           int64
resting_ecg            str
max_hr               int64
exercise_angina      int64
oldpeak            float64
st_slope               str
heart_disease        int64
dtype: object

Missing Values:
age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64


In [2]:
df.head()

   age  sex  chest_pain_type  resting_bp  cholesterol  fasting_bs resting_ecg  max_hr  exercise_angina  oldpeak st_slope  heart_disease
0   68    0  atypical_angina       138.0        214.0           0      normal     130                0      0.4       up              1
1   58    1      non_anginal       100.0        248.0           0      normal     122                0      1.1       up              1
2   44    1      non_anginal       110.0        197.0           0      normal     177                0      0.2       up              0
3   72    1     asymptomatic       125.0        238.0           1      normal     121                1      1.0       up              1
4   37    1      non_anginal       130.0          0.0           0      normal     187                0      0.4     flat              0

## 2. Exploratory Data Analysis

In [3]:
# Target class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Target distribution
ax1 = axes[0]
df['heart_disease'].value_counts().plot(kind='bar', ax=ax1, color=['steelblue', 'tomato'], edgecolor='black')
ax1.set_title('Target Class Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Heart Disease (0=Absent, 1=Present)')
ax1.set_ylabel('Count')
ax1.set_xticklabels(['Absent (0)', 'Present (1)'], rotation=0)
for p in ax1.patches:
    ax1.annotate(str(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()+5), ha='center')

# Plot 2: Correlation heatmap
ax2 = axes[1]
numerical_df = df.select_dtypes(include='number')
corr_matrix = numerical_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax2, linewidths=0.5)
ax2.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

# Plot 3: Age distribution by heart disease
ax3 = axes[2]
df[df['heart_disease']==0]['age'].hist(ax=ax3, alpha=0.6, label='No Disease', color='steelblue', bins=20)
df[df['heart_disease']==1]['age'].hist(ax=ax3, alpha=0.6, label='Disease', color='tomato', bins=20)
ax3.set_title('Age Distribution by Heart Disease', fontsize=13, fontweight='bold')
ax3.set_xlabel('Age')
ax3.set_ylabel('Frequency')
ax3.legend()

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plots saved.')

Plots saved.


**Interpretation:**
- **Class Distribution**: The dataset is nearly balanced — 393 patients without heart disease (49.1%) and 407 with (50.9%). No class imbalance correction is needed.
- **Correlation Heatmap**: `age` (0.29) and `exercise_angina` (0.29) show the strongest positive correlations with heart disease. `max_hr` shows a slight negative relationship, suggesting higher max heart rate is associated with healthier outcomes.
- **Age Distribution**: Patients with heart disease tend to be older (peak around 60–70), while disease-free patients are more evenly spread across age groups.

## 3. Data Preprocessing

In [4]:
# Handle missing values via median imputation
# Median is more robust than mean for skewed distributions
df_proc = df.copy()
df_proc['resting_bp'] = df_proc['resting_bp'].fillna(df_proc['resting_bp'].median())
df_proc['cholesterol'] = df_proc['cholesterol'].fillna(df_proc['cholesterol'].median())

print('Missing values after imputation:')
print(df_proc.isnull().sum())
print(f"\nImputed resting_bp with median: {df['resting_bp'].median():.1f}")
print(f"Imputed cholesterol with median: {df['cholesterol'].median():.1f}")

Missing values after imputation:
age                 0
sex                 0
chest_pain_type     0
resting_bp          0
cholesterol         0
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64

Imputed resting_bp with median: 130.0
Imputed cholesterol with median: 225.5


**Imputation Strategy:** Median imputation was chosen over mean because clinical measurements like `resting_bp` and `cholesterol` can be skewed by outliers. The median is a more stable estimate of the central tendency in such cases. Row-drop was avoided since we only have 800 records — dropping 56 rows (7%) would meaningfully reduce training data.

In [5]:
# One-hot encode categorical variables
cat_cols = ['chest_pain_type', 'resting_ecg', 'st_slope']
df_proc = pd.get_dummies(df_proc, columns=cat_cols, drop_first=False)
df_proc = df_proc.astype(float)

# Scale numerical features
num_cols = ['age', 'resting_bp', 'cholesterol', 'max_hr', 'oldpeak']
X = df_proc.drop('heart_disease', axis=1)
y = df_proc['heart_disease']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'\nFeatures after encoding: {list(X.columns)}')

Training set: (640, 18)
Test set:     (160, 18)

Features after encoding: ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs', 'max_hr', 'exercise_angina', 'oldpeak', 'chest_pain_type_asymptomatic', 'chest_pain_type_atypical_angina', 'chest_pain_type_non_anginal', 'chest_pain_type_typical_angina', 'resting_ecg_left_ventricular_hypertrophy', 'resting_ecg_normal', 'resting_ecg_st_t_wave_abnormality', 'st_slope_down', 'st_slope_flat', 'st_slope_up']


## 4. Model Training

In [6]:
# Train three classifiers with random_state=42
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)
gb = GradientBoostingClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

print('All three models trained successfully.')
print('  - DecisionTreeClassifier')
print('  - RandomForestClassifier')
print('  - GradientBoostingClassifier')

All three models trained successfully.
  - DecisionTreeClassifier
  - RandomForestClassifier
  - GradientBoostingClassifier


## 5. Model Evaluation

In [7]:
models = {
    'Decision Tree': dt,
    'Random Forest': rf,
    'Gradient Boosting': gb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'{'='*45}')
    print(f'  {name}')
    print(f'{'='*45}')
    cm = confusion_matrix(y_test, y_pred)
    print(f'Confusion Matrix:\n{cm}')
    print()
    print(classification_report(y_test, y_pred, target_names=['No Disease','Has Disease']))


  Decision Tree
Confusion Matrix:
[[56 23]
 [22 59]]

              precision    recall  f1-score   support

  No Disease       0.72      0.71      0.71        79
 Has Disease       0.72      0.73      0.72        81

    accuracy                           0.72       160
   macro avg       0.72      0.72      0.72       160
weighted avg       0.72      0.72      0.72       160

  Random Forest
Confusion Matrix:
[[60 19]
 [15 66]]

              precision    recall  f1-score   support

  No Disease       0.80      0.76      0.78        79
 Has Disease       0.78      0.81      0.80        81

    accuracy                           0.79       160
   macro avg       0.79      0.79      0.79       160
weighted avg       0.79      0.79      0.79       160

  Gradient Boosting
Confusion Matrix:
[[61 18]
 [18 63]]

              precision    recall  f1-score   support

  No Disease       0.77      0.77      0.77        79
 Has Disease       0.78      0.78      0.78        81

    accuracy    

**Best Model: Random Forest**

Random Forest outperforms both Decision Tree and Gradient Boosting on the test set:
- **Accuracy: 79%** vs 72% (DT) and 78% (GB)
- **F1-score (Has Disease): 0.80** — the highest among all three. In a medical setting, false negatives (missing actual disease) are costly; the Random Forest's higher recall for the positive class (0.81) makes it preferable.
- **Gradient Boosting** is close but slightly behind on recall for disease detection.
- **Decision Tree** overfits and generalises poorly despite being interpretable.

The Random Forest benefits from ensemble averaging over 100 trees, which reduces variance without sacrificing much bias.

## 6. Hyperparameter Tuning

In [8]:
# Tune Random Forest using GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print('Best Parameters Found:')
print(grid_search.best_params_)
print(f'Best CV F1 Score: {grid_search.best_score_:.4f}')

best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

print('\n--- Tuned Random Forest Test Performance ---')
print(classification_report(y_test, y_pred_tuned, target_names=['No Disease','Has Disease']))

y_pred_base = rf.predict(X_test)
from sklearn.metrics import f1_score, accuracy_score
print(f'Baseline RF  -> Accuracy: {accuracy_score(y_test, y_pred_base):.4f}, F1: {f1_score(y_test, y_pred_base):.4f}')
print(f'Tuned RF     -> Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}, F1: {f1_score(y_test, y_pred_tuned):.4f}')

Best Parameters Found:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1 Score: 0.7965

--- Tuned Random Forest Test Performance ---
              precision    recall  f1-score   support

  No Disease       0.80      0.76      0.78        79
 Has Disease       0.78      0.81      0.80        81

    accuracy                           0.79       160
   macro avg       0.79      0.79      0.79       160
weighted avg       0.79      0.79      0.79       160

Baseline RF  -> Accuracy: 0.7938, F1: 0.7988
Tuned RF     -> Accuracy: 0.7938, F1: 0.7988


**Tuning Result:** GridSearchCV confirmed that the default Random Forest hyperparameters (`n_estimators=100`, `max_depth=None`, `min_samples_split=2`) were already near-optimal for this dataset. The tuned model achieves an F1 of **0.80** on the test set — consistent with the baseline. This tells us the model is not being hampered by poor hyperparameter choices; the performance ceiling here is primarily driven by the features available in the data.